# Classificação de Defeitos em Módulos Fotovoltaicos por Imagens Termográficas

**Trabalho de Conclusão de Curso — Engenharia Elétrica**

---

## Descrição do Pipeline

Este notebook implementa o pipeline completo de treinamento, avaliação e otimização de
modelos de classificação binária (Normal vs Defeito) para imagens termográficas de painéis
fotovoltaicos, visando implantação em dispositivos embarcados (Raspberry Pi).

**Etapas:**

| # | Etapa | Descrição |
|---|-------|-----------|
| 1 | Divisão do Dataset | Split estratificado 70/15/15 |
| 2 | Pré-processamento | Equalização + Colormap INFERNO + Augmentation |
| 3 | Treinamento | Transfer learning em duas fases (head → fine-tuning) |
| 4 | Avaliação | Métricas no conjunto de teste |
| 5 | Quantização | Conversão para TFLite (Float16 e INT8) |
| 6 | Benchmark | Avaliação dos modelos quantizados |

**Arquiteturas comparadas:** MobileNetV2, EfficientNet-B0, MobileNetV3-Small

> **Nota:** Se o notebook for interrompido, as etapas seguintes recuperam resultados
> automaticamente do disco.

---
## 1. Configurações e Imports

In [ ]:
import os
import cv2
import shutil
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from pathlib import Path
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
)

# ── Reprodutibilidade ────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ── Hiperparâmetros globais ──────────────────────────────────────
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
AUTOTUNE   = tf.data.AUTOTUNE

# ── Diretórios ───────────────────────────────────────────────────
SOURCE_DIR  = 'data'         # data/normal/ e data/defect/
DATA_DIR    = 'data_split'   # Gerado pelo split
MODELS_DIR  = 'models'
RESULTS_DIR = 'results'

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# ── Informações do ambiente ──────────────────────────────────────
print(f'TensorFlow: {tf.__version__}')
print(f'Keras:      {tf.keras.__version__}')
print(f'GPU:        {bool(tf.config.list_physical_devices("GPU"))}')

---
## 2. Configurações por Arquitetura

Cada arquitetura possui hiperparâmetros otimizados individualmente para maximizar
a acurácia respeitando as restrições de tamanho do modelo.

In [ ]:
ARCH_CONFIGS = {
    'mobilenetv2': {
        'epochs_phase1':   10,
        'epochs_phase2':   30,
        'unfreeze_layers': 40,
        'head':            'melhorado',
        'dropout_1':       0.4,
        'dropout_2':       0.3,
        'lr_phase1':       1e-3,
        'lr_phase2':       1e-5,
        'weight_decay':    1e-4,
        'es_patience':     6,
        'augment_level':   'normal',
    },
    'efficientnetb0': {
        'epochs_phase1':   10,
        'epochs_phase2':   25,
        'unfreeze_layers': 35,
        'head':            'simples',
        'dropout_1':       0.3,
        'dropout_2':       None,
        'lr_phase1':       1e-3,
        'lr_phase2':       1e-5,
        'weight_decay':    1e-4,
        'es_patience':     5,
        'augment_level':   'normal',
    },
    'mobilenetv3small': {
        'epochs_phase1':   15,
        'epochs_phase2':   20,
        'unfreeze_layers': 10,
        'head':            'simples',
        'dropout_1':       0.3,
        'dropout_2':       None,
        'lr_phase1':       1e-4,
        'lr_phase2':       5e-6,
        'weight_decay':    5e-5,
        'es_patience':     5,
        'augment_level':   'leve',
    },
}

ARCHITECTURES = list(ARCH_CONFIGS.keys())
print(f'Arquiteturas configuradas: {ARCHITECTURES}')

---
## 3. Divisão do Dataset

Divisão estratificada: **70% Treino | 15% Validação | 15% Teste**

A divisão é executada apenas uma vez. Se a pasta `data_split` já existir, esta célula
será ignorada automaticamente.

In [ ]:
def split_dataset(
    source_dir: str,
    output_dir: str,
    splits: tuple = (0.70, 0.15, 0.15),
) -> None:
    """Divide imagens em treino/validação/teste mantendo proporção por classe."""
    random.seed(SEED)
    classes = ['normal', 'defect']

    for class_name in classes:
        class_path = Path(source_dir, class_name)
        if not class_path.exists():
            print(f'  [AVISO] Pasta não encontrada: {class_path}')
            continue

        images = sorted(class_path.glob('*.*'))
        random.shuffle(images)
        n = len(images)

        if n == 0:
            print(f'  [AVISO] Nenhuma imagem em: {class_path}')
            continue

        n_train = int(n * splits[0])
        n_val   = int(n * splits[1])

        subsets = {
            'train': images[:n_train],
            'val':   images[n_train:n_train + n_val],
            'test':  images[n_train + n_val:],
        }

        for subset, files in subsets.items():
            dest = Path(output_dir, subset, class_name)
            dest.mkdir(parents=True, exist_ok=True)
            for f in files:
                shutil.copy(f, dest / f.name)

        print(
            f'  {class_name:8s} → '
            f'{len(subsets["train"]):5d} treino | '
            f'{len(subsets["val"]):5d} val | '
            f'{len(subsets["test"]):5d} teste'
        )


if not os.path.exists(DATA_DIR):
    print('Dividindo dataset...')
    split_dataset(SOURCE_DIR, DATA_DIR)
    print('\nSplit concluído!')
else:
    # Mostra contagem existente
    print(f'Pasta "{DATA_DIR}" já existe — split ignorado.')
    for subset in ['train', 'val', 'test']:
        subset_path = Path(DATA_DIR, subset)
        if subset_path.exists():
            total = sum(1 for _ in subset_path.rglob('*.*'))
            print(f'  {subset:5s}: {total} imagens')

---
## 4. Pré-processamento e Data Augmentation

O pipeline de pré-processamento converte imagens térmicas em escala de cinza para
representação RGB usando:

1. **Normalização** — `cv2.normalize` para range [0, 255]
2. **Equalização de histograma** — melhora contraste
3. **Colormap INFERNO** — mapeia intensidades para escala de cores térmica

Dois níveis de augmentation são aplicados conforme a arquitetura:
- **Normal** — flip, rotação (±25°), zoom (12%), brilho e contraste (±15%)
- **Leve** — flip, rotação (±10°), zoom (5%), brilho (±10%)

In [ ]:
# ── Camadas de Augmentation ──────────────────────────────────────
augment_normal = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal_and_vertical'),
    tf.keras.layers.RandomRotation(0.25),
    tf.keras.layers.RandomZoom(0.12),
    tf.keras.layers.RandomBrightness(0.15),
    tf.keras.layers.RandomContrast(0.15),
], name='augmentation_normal')

augment_leve = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal_and_vertical'),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.05),
    tf.keras.layers.RandomBrightness(0.1),
], name='augmentation_leve')


def apply_colormap(img_np: np.ndarray) -> np.ndarray:
    """Aplica equalização de histograma + colormap INFERNO em imagem grayscale."""
    img_gray  = img_np[:, :, 0].astype(np.uint8)
    img_norm  = cv2.normalize(img_gray, None, 0, 255, cv2.NORM_MINMAX)
    img_eq    = cv2.equalizeHist(img_norm)
    img_color = cv2.applyColorMap(img_eq, cv2.COLORMAP_INFERNO)
    return cv2.cvtColor(img_color, cv2.COLOR_BGR2RGB).astype(np.float32)


def preprocess_thermal(
    image: tf.Tensor, label: tf.Tensor
) -> tuple:
    """Aplica pré-processamento térmico em batch de imagens."""
    def process_single(img):
        img_rgb = tf.numpy_function(apply_colormap, [img], tf.float32)
        img_rgb.set_shape((IMG_SIZE[0], IMG_SIZE[1], 3))
        return img_rgb

    image = tf.map_fn(process_single, image, fn_output_signature=tf.float32)
    return image, label


def load_dataset(subset: str, augment_level: str = None) -> tf.data.Dataset:
    """Carrega dataset de imagens com pré-processamento e augmentation opcional.

    Args:
        subset: 'train', 'val' ou 'test'
        augment_level: None, 'normal' ou 'leve'

    Returns:
        tf.data.Dataset pré-processado e otimizado.
    """
    subset_path = os.path.join(DATA_DIR, subset)
    if not os.path.exists(subset_path):
        raise FileNotFoundError(
            f'Diretório não encontrado: {subset_path}. '
            f'Execute a célula de split primeiro.'
        )

    ds = tf.keras.utils.image_dataset_from_directory(
        subset_path,
        labels='inferred',
        label_mode='binary',
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=(subset == 'train'),
        seed=SEED,
        color_mode='grayscale',
        interpolation='bicubic',
    )

    ds = ds.map(preprocess_thermal, num_parallel_calls=AUTOTUNE)

    if augment_level == 'normal':
        ds = ds.map(
            lambda x, y: (augment_normal(x, training=True), y),
            num_parallel_calls=AUTOTUNE,
        )
    elif augment_level == 'leve':
        ds = ds.map(
            lambda x, y: (augment_leve(x, training=True), y),
            num_parallel_calls=AUTOTUNE,
        )

    return ds.prefetch(AUTOTUNE)

In [ ]:
# ── Verificação visual do pipeline ──────────────────────────────
val_ds  = load_dataset('val')
test_ds = load_dataset('test')

train_check = load_dataset('train', augment_level='normal')

plt.figure(figsize=(14, 3))
for images, labels in train_check.take(1):
    for i in range(min(8, len(images))):
        plt.subplot(1, 8, i + 1)
        plt.imshow(images[i].numpy().astype(np.uint8))
        lbl = 'Defeito' if int(labels[i]) == 1 else 'Normal'
        plt.title(lbl, fontsize=7)
        plt.axis('off')

plt.suptitle(
    'Verificação do Pipeline — INFERNO + Equalização + Augmentation',
    fontweight='bold',
)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'verificacao_pipeline.png'), dpi=150)
plt.show()

del train_check
print('Datasets carregados com sucesso.')

---
## 5. Definição das Arquiteturas

Todas as arquiteturas utilizam `Rescaling` nativa do Keras (compatível com Keras ≥ 3.x)
ao invés de `preprocess_input`, evitando erros de serialização.

Dois tipos de cabeça de classificação:
- **Melhorado**: GAP → Dense(256) → BN → Dropout → Dense(64) → Dropout → Sigmoid
- **Simples**: GAP → Dropout → Sigmoid

In [ ]:
def build_model(
    arch_name: str, cfg: dict
) -> tuple:
    """Constrói modelo com transfer learning para classificação binária.

    Args:
        arch_name: Nome da arquitetura ('mobilenetv2', 'efficientnetb0', 'mobilenetv3small')
        cfg: Dicionário de configuração da arquitetura

    Returns:
        Tupla (modelo_completo, modelo_base) para controle de fine-tuning.
    """
    inputs = tf.keras.Input(shape=(*IMG_SIZE, 3), name='input_image')

    if arch_name == 'mobilenetv2':
        x = tf.keras.layers.Rescaling(scale=1.0 / 127.5, offset=-1.0)(inputs)
        base = tf.keras.applications.MobileNetV2(
            input_shape=(*IMG_SIZE, 3), include_top=False, weights='imagenet'
        )
    elif arch_name == 'efficientnetb0':
        x = inputs  # EfficientNet já inclui normalização interna
        base = tf.keras.applications.EfficientNetB0(
            input_shape=(*IMG_SIZE, 3), include_top=False, weights='imagenet'
        )
    elif arch_name == 'mobilenetv3small':
        x = tf.keras.layers.Rescaling(scale=1.0 / 127.5, offset=-1.0)(inputs)
        base = tf.keras.applications.MobileNetV3Small(
            input_shape=(*IMG_SIZE, 3), include_top=False, weights='imagenet'
        )
    else:
        raise ValueError(f'Arquitetura desconhecida: {arch_name}')

    base.trainable = False
    x = base(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)

    if cfg['head'] == 'melhorado':
        x = tf.keras.layers.Dense(256, activation='relu')(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Dropout(cfg['dropout_1'])(x)
        x = tf.keras.layers.Dense(64, activation='relu')(x)
        x = tf.keras.layers.Dropout(cfg['dropout_2'])(x)
    else:
        x = tf.keras.layers.Dropout(cfg['dropout_1'])(x)

    outputs = tf.keras.layers.Dense(1, activation='sigmoid', name='output')(x)

    model = tf.keras.Model(inputs, outputs, name=arch_name)
    return model, base

---
## 6. Funções de Treinamento, Avaliação e Quantização

In [ ]:
def train_model(arch_name: str) -> tuple:
    """Treina modelo em duas fases: head congelada → fine-tuning.

    Args:
        arch_name: Nome da arquitetura

    Returns:
        Tupla (modelo_treinado, dicionário_de_histórico)
    """
    cfg = ARCH_CONFIGS[arch_name]

    print(f"\n{'=' * 65}")
    print(f'  TREINANDO: {arch_name.upper()}')
    print(f"  Head: {cfg['head']} | Augmentation: {cfg['augment_level']} | "
          f"Épocas: {cfg['epochs_phase1']}+{cfg['epochs_phase2']} | "
          f"Unfreeze: {cfg['unfreeze_layers']} camadas")
    print(f"{'=' * 65}")

    # Carrega datasets frescos para cada arquitetura
    train_ds = load_dataset('train', augment_level=cfg['augment_level'])
    val_ds_local = load_dataset('val')

    model, base = build_model(arch_name, cfg)

    save_path = os.path.join(MODELS_DIR, f'{arch_name}.keras')
    best_weights_path = os.path.join(MODELS_DIR, f'{arch_name}_best.weights.h5')

    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(
            best_weights_path,
            monitor='val_accuracy',
            save_best_only=True,
            save_weights_only=True,
            verbose=1,
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor='val_accuracy',
            patience=cfg['es_patience'],
            restore_best_weights=True,
            verbose=1,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=3,
            min_lr=1e-7,
            verbose=1,
        ),
    ]

    # ── Fase 1: Treinamento da cabeça (base congelada) ────────────
    print(f'\n[FASE 1] Treinamento da cabeça — LR = {cfg["lr_phase1"]}')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=cfg['lr_phase1']),
        loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=0.1),
        metrics=['accuracy'],
    )
    h1 = model.fit(
        train_ds,
        validation_data=val_ds_local,
        epochs=cfg['epochs_phase1'],
        callbacks=callbacks,
    )

    # ── Fase 2: Fine-tuning (últimas N camadas descongeladas) ─────
    print(f"\n[FASE 2] Fine-tuning — últimas {cfg['unfreeze_layers']} camadas | LR = {cfg['lr_phase2']}")
    base.trainable = True
    for layer in base.layers[:-cfg['unfreeze_layers']]:
        layer.trainable = False

    model.compile(
        optimizer=tf.keras.optimizers.AdamW(
            learning_rate=cfg['lr_phase2'],
            weight_decay=cfg['weight_decay'],
        ),
        loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=0.1),
        metrics=['accuracy'],
    )
    h2 = model.fit(
        train_ds,
        validation_data=val_ds_local,
        epochs=cfg['epochs_phase2'],
        callbacks=callbacks,
    )

    # Salva modelo final
    model.save(save_path)
    print(f'\nModelo salvo em: {save_path}')

    # Combina histórico das duas fases
    history = {}
    for key in h1.history:
        history[key] = h1.history[key] + h2.history[key]

    pd.DataFrame(history).to_csv(
        os.path.join(RESULTS_DIR, f'history_{arch_name}.csv'), index=False
    )

    return model, history

In [ ]:
def evaluate_model(arch_name: str) -> tuple:
    """Avalia modelo no conjunto de teste e gera matriz de confusão.

    Args:
        arch_name: Nome da arquitetura

    Returns:
        Tupla (acurácia, relatório_classificação)
    """
    model_path = os.path.join(MODELS_DIR, f'{arch_name}.keras')
    if not os.path.exists(model_path):
        raise FileNotFoundError(
            f'Modelo não encontrado: {model_path}. Execute a Etapa de Treinamento primeiro.'
        )

    model = tf.keras.models.load_model(model_path)
    test_ds_local = load_dataset('test')

    y_true, y_pred_prob = [], []
    for images, labels in test_ds_local:
        probs = model.predict(images, verbose=0)
        y_pred_prob.extend(probs.flatten())
        y_true.extend(labels.numpy().flatten())

    y_true = np.array(y_true, dtype=int)
    y_pred_prob = np.array(y_pred_prob)
    y_pred = (y_pred_prob > 0.5).astype(int)

    report = classification_report(
        y_true, y_pred,
        target_names=['Normal', 'Defeito'],
        output_dict=True,
    )
    accuracy = report['accuracy']

    # Exibe relatório
    print(f"\n{'─' * 50}")
    print(f'  {arch_name.upper()} — Acurácia: {accuracy * 100:.2f}%')
    print(f"{'─' * 50}")
    print(classification_report(y_true, y_pred, target_names=['Normal', 'Defeito']))

    # Matriz de confusão
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=['Normal', 'Defeito'],
        yticklabels=['Normal', 'Defeito'],
    )
    plt.title(f'Matriz de Confusão — {arch_name}')
    plt.ylabel('Real')
    plt.xlabel('Predito')
    plt.tight_layout()
    plt.savefig(
        os.path.join(RESULTS_DIR, f'confusion_matrix_{arch_name}.png'), dpi=150
    )
    plt.show()

    del model
    tf.keras.backend.clear_session()

    return accuracy, report

In [ ]:
def get_representative_dataset():
    """Gera dataset representativo para calibração da quantização INT8."""
    def generator():
        cal_ds = load_dataset('val')
        count = 0
        for images, _ in cal_ds:
            if count >= 100:
                break
            yield [tf.cast(images, tf.float32)]
            count += 1
    return generator


def quantize_model(arch_name: str) -> tuple:
    """Quantiza modelo Keras para TFLite (Float16 e INT8).

    Args:
        arch_name: Nome da arquitetura

    Returns:
        Tupla (path_f16, path_int8, size_full, size_f16, size_int8)
    """
    model_path = os.path.join(MODELS_DIR, f'{arch_name}.keras')
    if not os.path.exists(model_path):
        raise FileNotFoundError(
            f'Modelo não encontrado: {model_path}. Execute o treinamento primeiro.'
        )

    print(f"\n{'─' * 55}")
    print(f'  Quantizando: {arch_name.upper()}')

    model = tf.keras.models.load_model(model_path)
    size_full = os.path.getsize(model_path) / 1e6

    # ── Float16 ──────────────────────────────────────────────────
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.target_spec.supported_types = [tf.float16]
    tflite_f16 = converter.convert()

    path_f16 = os.path.join(MODELS_DIR, f'{arch_name}_f16.tflite')
    with open(path_f16, 'wb') as f:
        f.write(tflite_f16)
    print(f'  [Float16] Salvo: {path_f16}')

    # ── INT8 (full integer quantization) ─────────────────────────
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = get_representative_dataset()
    converter.target_spec.supported_ops = [
        tf.lite.OpsSet.TFLITE_BUILTINS_INT8,
        tf.lite.OpsSet.TFLITE_BUILTINS,
    ]
    converter.inference_input_type = tf.float32
    converter.inference_output_type = tf.float32
    tflite_int8 = converter.convert()

    path_int8 = os.path.join(MODELS_DIR, f'{arch_name}_int8.tflite')
    with open(path_int8, 'wb') as f:
        f.write(tflite_int8)
    print(f'  [INT8]    Salvo: {path_int8}')

    # ── Resumo de tamanhos ────────────────────────────────────────
    size_f16 = len(tflite_f16) / 1e6
    size_int8 = len(tflite_int8) / 1e6
    print(
        f'  Tamanhos → Full: {size_full:.2f} MB | '
        f'F16: {size_f16:.2f} MB ({size_f16 / size_full * 100:.0f}%) | '
        f'INT8: {size_int8:.2f} MB ({size_int8 / size_full * 100:.0f}%)'
    )

    del model
    tf.keras.backend.clear_session()

    return path_f16, path_int8, size_full, size_f16, size_int8

In [ ]:
def make_interpreter(model_path: str):
    """Cria interpretador TFLite com fallback entre ai_edge_litert e tf.lite."""
    try:
        from ai_edge_litert.interpreter import Interpreter
        return Interpreter(model_path=model_path)
    except ImportError:
        pass
    try:
        return tf.lite.Interpreter(model_path=model_path, num_threads=4)
    except TypeError:
        return tf.lite.Interpreter(model_path=model_path)


def evaluate_tflite(model_path: str, label: str) -> float:
    """Avalia modelo TFLite no conjunto de teste.

    Args:
        model_path: Caminho para o arquivo .tflite
        label: Rótulo para exibição ('Float16' ou 'INT8')

    Returns:
        Acurácia em porcentagem.
    """
    print(f'  Avaliando [{label}]...')

    interpreter = make_interpreter(model_path)
    interpreter.allocate_tensors()
    inp_details = interpreter.get_input_details()
    out_details = interpreter.get_output_details()

    test_ds_local = load_dataset('test')
    y_true, y_pred = [], []

    for images, labels_batch in test_ds_local:
        for i in range(len(images)):
            img = np.expand_dims(images[i].numpy(), 0).astype(np.float32)
            interpreter.set_tensor(inp_details[0]['index'], img)
            interpreter.invoke()
            prob = float(interpreter.get_tensor(out_details[0]['index'])[0][0])
            y_pred.append(1 if prob > 0.5 else 0)
            y_true.append(int(labels_batch[i].numpy()))

    acc = accuracy_score(y_true, y_pred) * 100
    print(f'  Acurácia [{label}]: {acc:.2f}%')
    return acc


print('Funções de treinamento, avaliação e quantização definidas.')

---
## 7. Etapa 1 — Treinamento

> Para treinar apenas uma arquitetura específica, edite `ARCHITECTURES` na célula de configuração.
>
> Exemplo: `ARCHITECTURES = ['mobilenetv2']`

In [ ]:
trained_history = {}

for arch in ARCHITECTURES:
    _, history = train_model(arch)
    trained_history[arch] = history
    tf.keras.backend.clear_session()

print('\nEtapa 1 (Treinamento) concluída!')

---
## 8. Etapa 2 — Avaliação no Conjunto de Teste

In [ ]:
eval_results = {}

for arch in ARCHITECTURES:
    acc, report = evaluate_model(arch)
    eval_results[arch] = {
        'accuracy': acc,
        'history':  trained_history.get(arch, {}),
        'report':   report,
    }

# ── Tabela resumo ────────────────────────────────────────────────
summary = pd.DataFrame([
    {
        'arquitetura':      arch,
        'acuracia_teste':   round(data['accuracy'] * 100, 2),
        'precisao_normal':  round(data['report']['Normal']['precision'] * 100, 2),
        'recall_normal':    round(data['report']['Normal']['recall'] * 100, 2),
        'precisao_defeito': round(data['report']['Defeito']['precision'] * 100, 2),
        'recall_defeito':   round(data['report']['Defeito']['recall'] * 100, 2),
    }
    for arch, data in eval_results.items()
])

summary.to_csv(os.path.join(RESULTS_DIR, 'resumo_comparativo.csv'), index=False)
print('\nResumo comparativo:')
display(summary)

In [ ]:
# ── Gráfico comparativo de acurácia ─────────────────────────────
archs_com_historico = [
    a for a in ARCHITECTURES if eval_results.get(a, {}).get('history')
]

if archs_com_historico:
    colors = ['#2196F3', '#4CAF50', '#FF9800']
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Curvas de aprendizado
    ax1 = axes[0]
    for i, name in enumerate(archs_com_historico):
        data = eval_results[name]
        epochs = range(1, len(data['history']['accuracy']) + 1)
        ax1.plot(
            epochs, data['history']['accuracy'],
            label=f'{name} (treino)', color=colors[i], linewidth=2,
        )
        ax1.plot(
            epochs, data['history']['val_accuracy'],
            label=f'{name} (val)', color=colors[i],
            linewidth=2, linestyle='--',
        )
    ax1.set_title('Acurácia por Época')
    ax1.set_xlabel('Época')
    ax1.set_ylabel('Acurácia')
    ax1.legend(fontsize=8)
    ax1.grid(True, alpha=0.3)

    # Barras de acurácia final
    ax2 = axes[1]
    accs = [eval_results[n]['accuracy'] * 100 for n in ARCHITECTURES]
    bars = ax2.bar(
        ARCHITECTURES, accs,
        color=colors[:len(ARCHITECTURES)],
        width=0.5, edgecolor='white',
    )
    ax2.set_title('Acurácia Final — Conjunto de Teste')
    ax2.set_ylabel('Acurácia (%)')
    ax2.set_ylim([max(0, min(accs) - 5), 100])
    for bar, acc in zip(bars, accs):
        ax2.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.3,
            f'{acc:.2f}%',
            ha='center', va='bottom', fontweight='bold',
        )
    ax2.grid(True, axis='y', alpha=0.3)

    plt.suptitle(
        'Comparativo de Arquiteturas — Painéis Fotovoltaicos',
        fontsize=11, fontweight='bold',
    )
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'comparativo_acuracia.png'), dpi=150)
    plt.show()
else:
    print(
        'Histórico não disponível — execute a Etapa de Treinamento '
        'para gerar o gráfico de curvas.'
    )

---
## 9. Etapa 3 — Quantização para TFLite

Converte os modelos treinados para formato otimizado para dispositivos embarcados:
- **Float16** — reduz tamanho pela metade, sem perda significativa de acurácia
- **INT8** — reduz ~4x, ideal para inferência em CPU (Raspberry Pi)

In [ ]:
quant_results = {}

for arch in ARCHITECTURES:
    path_f16, path_int8, sf, sf16, si8 = quantize_model(arch)
    quant_results[arch] = {
        'path_f16':  path_f16,
        'path_int8': path_int8,
        'size_full': sf,
        'size_f16':  sf16,
        'size_int8': si8,
    }

print('\nEtapa 3 (Quantização) concluída!')

---
## 10. Etapa 4 — Avaliação dos Modelos Quantizados

Compara acurácia e tamanho dos modelos Full, Float16 e INT8 para validar que a
quantização não introduziu degradação inaceitável.

In [ ]:
# ── Recuperação de resultados anteriores (se necessário) ─────────
if not eval_results:
    csv_path = os.path.join(RESULTS_DIR, 'resumo_comparativo.csv')
    if os.path.exists(csv_path):
        df_prev = pd.read_csv(csv_path)
        for _, row in df_prev.iterrows():
            eval_results[row['arquitetura']] = {
                'accuracy': row['acuracia_teste'] / 100,
                'history': {},
                'report': {
                    'Normal': {
                        'precision': row['precisao_normal'] / 100,
                        'recall': row['recall_normal'] / 100,
                    },
                    'Defeito': {
                        'precision': row['precisao_defeito'] / 100,
                        'recall': row['recall_defeito'] / 100,
                    },
                },
            }
        print(f'eval_results recuperado do CSV: {list(eval_results.keys())}')

if not quant_results:
    for arch in ARCHITECTURES:
        path_f16 = os.path.join(MODELS_DIR, f'{arch}_f16.tflite')
        path_int8 = os.path.join(MODELS_DIR, f'{arch}_int8.tflite')
        model_path = os.path.join(MODELS_DIR, f'{arch}.keras')

        if not os.path.exists(path_f16) or not os.path.exists(path_int8):
            print(f'  [AVISO] TFLite de {arch} não encontrado — execute a Etapa 3.')
            continue

        size_full = os.path.getsize(model_path) / 1e6 if os.path.exists(model_path) else 0
        quant_results[arch] = {
            'path_f16': path_f16,
            'path_int8': path_int8,
            'size_full': size_full,
            'size_f16': os.path.getsize(path_f16) / 1e6,
            'size_int8': os.path.getsize(path_int8) / 1e6,
        }
    print(f'quant_results reconstruído: {list(quant_results.keys())}')

In [ ]:
# ── Avaliação TFLite ─────────────────────────────────────────────
quant_rows = []

for arch in ARCHITECTURES:
    if arch not in quant_results:
        print(f'  [AVISO] {arch} ausente — pulando.')
        continue

    print(f'\n  {arch.upper()}')
    acc_full = eval_results.get(arch, {}).get('accuracy', 0) * 100
    acc_f16 = evaluate_tflite(quant_results[arch]['path_f16'], 'Float16')
    acc_int8 = evaluate_tflite(quant_results[arch]['path_int8'], 'INT8')

    qr = quant_results[arch]
    quant_rows.append({
        'arquitetura':       arch,
        'tamanho_full_mb':   round(qr['size_full'], 2),
        'tamanho_f16_mb':    round(qr['size_f16'], 2),
        'tamanho_int8_mb':   round(qr['size_int8'], 2),
        'reducao_f16_pct':   round((1 - qr['size_f16'] / qr['size_full']) * 100, 1) if qr['size_full'] > 0 else 0,
        'reducao_int8_pct':  round((1 - qr['size_int8'] / qr['size_full']) * 100, 1) if qr['size_full'] > 0 else 0,
        'acuracia_full_pct': round(acc_full, 2),
        'acuracia_f16_pct':  round(acc_f16, 2),
        'acuracia_int8_pct': round(acc_int8, 2),
        'perda_f16_pct':     round(acc_full - acc_f16, 2),
        'perda_int8_pct':    round(acc_full - acc_int8, 2),
    })

df_quant = pd.DataFrame(quant_rows)
df_quant.to_csv(os.path.join(RESULTS_DIR, 'resumo_quantizacao.csv'), index=False)
print('\nResumo de quantização:')
display(df_quant)

In [ ]:
# ── Gráfico comparativo de quantização ───────────────────────────
if not df_quant.empty:
    colors_quant = {'Full': '#1565C0', 'F16': '#2E7D32', 'INT8': '#E65100'}
    archs = df_quant['arquitetura'].tolist()
    x = range(len(archs))
    w = 0.25

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Tamanho dos modelos
    ax1 = axes[0]
    b1 = ax1.bar([i - w for i in x], df_quant['tamanho_full_mb'], w, label='Full', color=colors_quant['Full'])
    b2 = ax1.bar([i for i in x], df_quant['tamanho_f16_mb'], w, label='Float16', color=colors_quant['F16'])
    b3 = ax1.bar([i + w for i in x], df_quant['tamanho_int8_mb'], w, label='INT8', color=colors_quant['INT8'])
    for bars in [b1, b2, b3]:
        for bar in bars:
            ax1.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.1,
                f'{bar.get_height():.1f} MB',
                ha='center', fontsize=8, fontweight='bold',
            )
    ax1.set_title('Tamanho dos Modelos')
    ax1.set_ylabel('MB')
    ax1.set_xticks(list(x))
    ax1.set_xticklabels(archs)
    ax1.legend()
    ax1.grid(True, axis='y', alpha=0.3)

    # Acurácia por formato
    ax2 = axes[1]
    b4 = ax2.bar([i - w for i in x], df_quant['acuracia_full_pct'], w, label='Full', color=colors_quant['Full'])
    b5 = ax2.bar([i for i in x], df_quant['acuracia_f16_pct'], w, label='Float16', color=colors_quant['F16'])
    b6 = ax2.bar([i + w for i in x], df_quant['acuracia_int8_pct'], w, label='INT8', color=colors_quant['INT8'])
    for bars in [b4, b5, b6]:
        for bar in bars:
            ax2.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.1,
                f'{bar.get_height():.1f}%',
                ha='center', fontsize=8, fontweight='bold',
            )
    min_acc = df_quant[['acuracia_full_pct', 'acuracia_f16_pct', 'acuracia_int8_pct']].min().min()
    ax2.set_title('Acurácia por Formato')
    ax2.set_ylabel('Acurácia (%)')
    ax2.set_ylim([max(0, min_acc - 5), 100])
    ax2.set_xticks(list(x))
    ax2.set_xticklabels(archs)
    ax2.legend()
    ax2.grid(True, axis='y', alpha=0.3)

    plt.suptitle(
        'Comparativo de Quantização — Full vs Float16 vs INT8',
        fontsize=12, fontweight='bold',
    )
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'comparativo_quantizacao.png'), dpi=150)
    plt.show()
else:
    print('Nenhum resultado de quantização disponível.')

---
## 11. Predição em Imagem Individual

Função utilitária para testar o modelo em uma imagem específica.
Suporta tanto modelos `.keras` quanto `.tflite`.

In [ ]:
def predict_single(
    model_path: str,
    image_path: str,
    threshold: float = 0.5,
    use_tflite: bool = False,
) -> tuple:
    """Realiza predição em uma única imagem termográfica.

    Args:
        model_path: Caminho para o modelo (.keras ou .tflite)
        image_path: Caminho para a imagem de entrada
        threshold: Limiar de decisão (padrão: 0.5)
        use_tflite: Se True, usa interpretador TFLite

    Returns:
        Tupla (rótulo_predito, probabilidade_raw)
    """
    # Pré-processamento
    img_raw = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if img_raw is None:
        raise FileNotFoundError(f'Imagem não encontrada: {image_path}')

    img_resz = cv2.resize(img_raw, IMG_SIZE, interpolation=cv2.INTER_CUBIC)
    img_norm = cv2.normalize(img_resz, None, 0, 255, cv2.NORM_MINMAX)
    img_eq = cv2.equalizeHist(img_norm)
    img_color = cv2.applyColorMap(img_eq, cv2.COLORMAP_INFERNO)
    img_rgb = cv2.cvtColor(img_color, cv2.COLOR_BGR2RGB).astype(np.float32)
    img_batch = np.expand_dims(img_rgb, axis=0)

    # Inferência
    if use_tflite:
        interpreter = make_interpreter(model_path)
        interpreter.allocate_tensors()
        inp = interpreter.get_input_details()
        out = interpreter.get_output_details()
        interpreter.set_tensor(inp[0]['index'], img_batch)
        interpreter.invoke()
        prob = float(interpreter.get_tensor(out[0]['index'])[0][0])
    else:
        model = tf.keras.models.load_model(model_path)
        prob = float(model.predict(img_batch, verbose=0)[0][0])

    # Resultado
    label = 'DEFEITO' if prob > threshold else 'NORMAL'
    conf = prob if prob > threshold else 1 - prob
    color_hex = '#e53935' if label == 'DEFEITO' else '#43a047'

    # Visualização
    plt.figure(figsize=(4, 4))
    plt.imshow(cv2.cvtColor(img_color, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.title(
        f'{label}  |  {conf:.1%}',
        fontsize=13, fontweight='bold', color=color_hex,
    )
    plt.tight_layout()
    plt.show()

    print(f'Resultado:  {label}')
    print(f'Confiança:  {conf:.1%}')
    print(f'Prob. raw:  {prob:.4f}')

    return label, prob


# ── Exemplos de uso (descomente para testar) ─────────────────────
# predict_single(
#     'models/mobilenetv2.keras',
#     'data_split/test/defect/exemplo.jpg',
# )
#
# predict_single(
#     'models/mobilenetv2_f16.tflite',
#     'data_split/test/defect/exemplo.jpg',
#     use_tflite=True,
# )

print('Função predict_single disponível.')

---
## 12. Resumo Final

Consolidação de todos os resultados e instruções para deploy.

In [ ]:
print('=' * 65)
print('  PIPELINE CONCLUÍDO COM SUCESSO')
print('=' * 65)
print(f'\n  Modelos Keras:   {MODELS_DIR}/<arch>.keras')
print(f'  Modelos TFLite:  {MODELS_DIR}/<arch>_f16.tflite  |  <arch>_int8.tflite')
print(f'  Resultados:      {RESULTS_DIR}/')
print(f'\n  Próximo passo — Deploy na Raspberry Pi:')
print(f'  $ scp models/*_int8.tflite pi@<IP>:~/models/')
print(f'  $ python inference_raspberry.py --test_dir data_split/test')

# ── Exibe resumos salvos ─────────────────────────────────────────
csv_train = os.path.join(RESULTS_DIR, 'resumo_comparativo.csv')
csv_quant = os.path.join(RESULTS_DIR, 'resumo_quantizacao.csv')

if os.path.exists(csv_train):
    print('\n  Resumo de Treinamento:')
    display(pd.read_csv(csv_train))

if os.path.exists(csv_quant):
    print('\n  Resumo de Quantização:')
    display(pd.read_csv(csv_quant))